# Simple RAG Training Pipeline

This notebook follows the same 4-step flow and uses the new CgftPipeline API:

1. **Point to dataset** - Chunk and upload documents
2. **Create synthetic QA** - Generate question-answer pairs with CgftPipeline
3. **Create env** - Load search environment
4. **Run training job** - Launch the training


## Setup

Install dependencies and configure API credentials.


In [ ]:
!pip install git+https://github.com/cgftinc/cgft.git

In [ ]:
!rm -rf posthog.com
!git clone --depth 1 --filter=blob:none --sparse https://github.com/PostHog/posthog.com.git
%cd posthog.com
!git sparse-checkout set contents/docs

In [ ]:
from cgft.utils import ensure_safe_python_version

ensure_safe_python_version()

In [ ]:
# Configuration

# create an API key from the cgft console (https://app.cgft.io/account/api-keys).
API_KEY = ""
BASE_URL = "https://app.cgft.io"
LLM_API_KEY = ""
LLM_BASE_URL = "https://expt-platform-foundry.openai.azure.com/openai/v1"

# Dataset configuration
DOCS_PATH = "./contents/docs/"
CORPUS_NAME = "posthog"

# Corpus intent used for corpus profiling (summary + example queries)
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation configuration (CgftPipeline)
TOTAL_SAMPLES = 100
OUTPUT_DIR = "outputs/cgft_simple_rag"

# Optional: name for your training experiment
EXPERIMENT_NAME = "posthog search v1"

# Generation/filter/refinement behavior follows library defaults.
# Override in the config cell below only when you intentionally want non-default behavior.

## Step 1: Point to Dataset

Chunk your documents and upload to corpus API in one line.


In [ ]:
from cgft.corpus.corpora.source import CorporaChunkSource

source = CorporaChunkSource(api_key=API_KEY, corpus_name=CORPUS_NAME, base_url=BASE_URL)
source.populate_from_folder(DOCS_PATH)

## Step 2: Create Synthetic QA

Generate synthetic QA pairs with `CgftPipeline` using the corpus from Step 1.


In [ ]:
from cgft.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    OutputConfig,
    PlatformConfig,
    TargetsConfig,
)
from cgft_utils.qa_generation.cgft_pipeline import CgftPipeline

cfg = CgftPipelineConfig(
    platform=PlatformConfig(api_key=API_KEY, base_url=BASE_URL, llm_api_key=LLM_API_KEY, llm_base_url=LLM_BASE_URL),
    corpus=CorpusConfig(corpus_name=CORPUS_NAME, min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        num_top_level_samples=4,
        num_random_samples=4,
        min_chunk_chars=400,
    ),
    targets=TargetsConfig(total_samples=TOTAL_SAMPLES),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
print("generator model:", cfg.generation.llm_direct.model)

In [ ]:
# Reuse the already-loaded corpus source from Step 1.
import importlib
from pprint import pprint

import cgft.qa_generation.cgft_pipeline as _cgft_pipeline_mod
import cgft.qa_generation.generators.direct_llm as _direct_llm_mod

# Force-refresh generation modules in case notebook state still has stale classes.
importlib.reload(_direct_llm_mod)
importlib.reload(_cgft_pipeline_mod)
CgftPipeline = _cgft_pipeline_mod.CgftPipeline

cgft_pipeline = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = cgft_pipeline.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
stats = result["stats"]


In [ ]:
for qa_pair in result['train_dataset'][:10]:
    print("Q:", qa_pair['question'])
    print("A:", qa_pair['answer'])

## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

In [ ]:
import cgft
from cgft.envs.cgft_search_env import CgftSearchEnv
from cgft.trainer.pipeline import train
experiment_id = train(
    env_class=CgftSearchEnv,
    env_args={
        "api_key": API_KEY,
        "corpus_id": source.corpus_id,
        "base_url": BASE_URL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix="cgft-search",
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[cgft],
    experiment_name=EXPERIMENT_NAME,
)


## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse
